# Proyecto 2 — Análisis Exploratorio
## Identificación de especies de mosquitos — AI Mosquito Alert Challenge 2023

**Curso:** CC3084 — Data Science  
**Universidad del Valle de Guatemala**

# 1. Investigación breve

El **AI Mosquito Alert Challenge 2023** busca mejorar la identificación automática de mosquitos mediante visión por computadora. La publicación oficial contiene imágenes etiquetadas y un *bounding box* por imagen.

La identificación de mosquitos es relevante para vigilancia y salud pública porque distintos grupos pueden participar en la transmisión de enfermedades. La OMS señala a **Aedes aegypti** como principal vector del dengue; CDC también reconoce grupos **Aedes, Culex y Anopheles** entre mosquitos capaces de transmitir patógenos. Este proyecto **no diagnostica enfermedades**: analiza un dataset de imágenes antes de desarrollar modelos de clasificación/detección.

### Fuentes
- Mosquito Alert / Zenodo: https://doi.org/10.5281/zenodo.15063886
- Hugging Face: https://huggingface.co/datasets/mosquito-alert/ai-mosquito-alert-challenge-2023
- OMS — Dengue: https://www.who.int/news-room/fact-sheets/detail/dengue-and-severe-dengue
- OMS — Enfermedades transmitidas por vectores: https://www.who.int/es/news-room/fact-sheets/detail/vector-borne-diseases
- CDC — Mosquitoes: https://www.cdc.gov/mosquitoes/
- CDC — Malaria: https://www.cdc.gov/malaria/

# 2. Preparación, descarga y carga

El notebook revisa si el dataset ya existe en la sesión. Si no existe, lo descarga automáticamente desde la publicación oficial en Hugging Face, lo descomprime y **borra el ZIP pesado** para liberar espacio.

In [1]:
from pathlib import Path
import os
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# --------------------------------------------------
# RUTAS TEMPORALES DE COLAB
# --------------------------------------------------
BASE = Path("/content/mosquitoalert")
DATASET = BASE / "dataset"
IMAGES = DATASET / "images"
LABELS = DATASET / "labels"

RESULTS = Path("/content/resultados_mosquitoalert")
FIGURES = RESULTS / "figuras"
TABLES = RESULTS / "tablas"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)


def dataset_esta_listo():
    if not IMAGES.exists() or not LABELS.exists():
        return False

    csvs_local = list(LABELS.glob("*.csv"))
    if not csvs_local:
        return False

    extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return any(
        p.is_file() and p.suffix.lower() in extensiones
        for p in IMAGES.iterdir()
    )


# --------------------------------------------------
# DESCARGA AUTOMÁTICA SI HACE FALTA
# --------------------------------------------------
if dataset_esta_listo():
    print("✅ Dataset ya disponible en esta sesión. No se volverá a descargar.")
else:
    print("Dataset no encontrado en /content. Se descargará ahora.")

    !pip install -q -U huggingface_hub
    from huggingface_hub import hf_hub_download

    BASE.mkdir(parents=True, exist_ok=True)

    downloaded = hf_hub_download(
        repo_id="mosquito-alert/ai-mosquito-alert-challenge-2023",
        filename="mosquito_dataset_ai_v1.zip",
        repo_type="dataset",
        local_dir=str(BASE)
    )

    print("ZIP descargado en:", downloaded)
    print("Descomprimiendo dataset...")

    DATASET.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(downloaded, "r") as z:
        z.extractall(DATASET)

    print("✅ Dataset descomprimido.")

    downloaded_path = Path(downloaded)
    if downloaded_path.exists():
        downloaded_path.unlink()
        print("✅ ZIP de aproximadamente 10 GB eliminado para liberar espacio.")

    if not dataset_esta_listo():
        raise RuntimeError(
            "La descarga terminó, pero no se detectó correctamente la estructura "
            "images/ y labels/."
        )


# --------------------------------------------------
# CARGAR CSV Y VERIFICAR DATASET
# --------------------------------------------------
csvs = list(LABELS.glob("*.csv"))

if not csvs:
    raise FileNotFoundError(f"No se encontró ningún CSV en {LABELS}")

CSV_PATH = csvs[0]
df = pd.read_csv(CSV_PATH)

extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_paths = [
    p for p in IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in extensiones
]

print("\nVERIFICACIÓN DEL DATASET")
print("CSV:", CSV_PATH.name)
print("Cantidad de imágenes:", len(image_paths))
print("Cantidad de registros:", len(df))
print("Variables originales:", df.shape[1])

if len(image_paths) == len(df):
    print("✅ La cantidad de imágenes coincide con la cantidad de registros.")
else:
    print("⚠️ La cantidad de imágenes y registros NO coincide.")

display(df.head(10))

Dataset no encontrado en /content. Se descargará ahora.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 9.4 MB/s eta 0:00:00


mosquito_dataset_ai_v1.zip: reconstructing file:   0%|          |  0.00B / 10.4GB            

mosquito_dataset_ai_v1.zip: downloading bytes:           |  0.00B            

ZIP descargado en: /content/mosquitoalert/mosquito_dataset_ai_v1.zip
Descomprimiendo dataset...
✅ Dataset descomprimido.
✅ ZIP de aproximadamente 10 GB eliminado para liberar espacio.

VERIFICACIÓN DEL DATASET
CSV: annotations.csv
Cantidad de imágenes: 10357
Cantidad de registros: 10357
Variables originales: 8
✅ La cantidad de imágenes coincide con la cantidad de registros.


,img_fName,img_w,img_h,bbx_xtl,bbx_ytl,bbx_xbr,bbx_ybr,class_label
0,train_00000.jpeg,2448,3264,1301,1546,1641,2096,albopictus
1,train_00001.jpeg,3024,4032,900,1897,1950,2990,albopictus
2,train_00002.jpeg,768,1024,220,58,659,808,albopictus
3,train_00003.jpeg,3456,4608,1169,2364,1586,2826,albopictus
4,train_00004.jpeg,1024,1365,129,231,697,1007,culex
5,train_00005.jpeg,1152,2560,198,798,954,1351,albopictus
6,train_00006.jpeg,3072,4080,1104,1030,2458,2911,anopheles
7,train_00007.jpeg,2128,4608,248,728,2049,1992,albopictus
8,train_00008.jpeg,4000,2250,1768,899,2414,1591,albopictus
9,train_00009.jpeg,768,1024,180,140,683,744,albopictus


# 3. Descripción de los datos

Variables originales:

| Variable | Descripción |
|---|---|
| `img_fName` | nombre del archivo |
| `img_w` | ancho de imagen |
| `img_h` | alto de imagen |
| `bbx_xtl` | X superior izquierda |
| `bbx_ytl` | Y superior izquierda |
| `bbx_xbr` | X inferior derecha |
| `bbx_ybr` | Y inferior derecha |
| `class_label` | clase del mosquito |

In [2]:
print("TIPOS DE VARIABLES")
df.info()

print("\nRESUMEN DE TIPOS")
display(pd.DataFrame({
    "variable": df.columns,
    "dtype": df.dtypes.astype(str).values
}))

TIPOS DE VARIABLES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10357 entries, 0 to 10356
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   img_fName    10357 non-null  object
 1   img_w        10357 non-null  int64 
 2   img_h        10357 non-null  int64 
 3   bbx_xtl      10357 non-null  int64 
 4   bbx_ytl      10357 non-null  int64 
 5   bbx_xbr      10357 non-null  int64 
 6   bbx_ybr      10357 non-null  int64 
 7   class_label  10357 non-null  object
dtypes: int64(6), object(2)
memory usage: 647.4+ KB

RESUMEN DE TIPOS


,variable,dtype
0,img_fName,object
1,img_w,int64
2,img_h,int64
3,bbx_xtl,int64
4,bbx_ytl,int64
5,bbx_xbr,int64
6,bbx_ybr,int64
7,class_label,object


# 4. Limpieza y control de calidad

Se revisan valores faltantes, duplicados, correspondencia entre CSV e imágenes, clases inesperadas y cajas inválidas. No se eliminan outliers automáticamente.

In [3]:
# Faltantes y duplicados
faltantes = df.isnull().sum().to_frame("faltantes")
faltantes["porcentaje"] = (faltantes["faltantes"] / len(df) * 100).round(3)

duplicados_filas = int(df.duplicated().sum())
duplicados_nombres = int(df["img_fName"].duplicated().sum())

print("VALORES FALTANTES")
display(faltantes)
print("Total faltantes:", int(df.isnull().sum().sum()))
print("Filas duplicadas:", duplicados_filas)
print("Nombres de imagen duplicados:", duplicados_nombres)

# Correspondencia CSV ↔ imágenes
image_files = [p.name for p in IMAGES.iterdir() if p.is_file()]
image_set = set(image_files)
csv_set = set(df["img_fName"].astype(str))

faltan_en_carpeta = sorted(csv_set - image_set)
sin_anotacion = sorted(image_set - csv_set)

print("\nImágenes en carpeta:", len(image_files))
print("Registros en CSV:", len(df))
print("Mencionadas en CSV pero inexistentes:", len(faltan_en_carpeta))
print("Imágenes sin anotación:", len(sin_anotacion))

# Clases esperadas
clases_esperadas = {
    "aegypti", "albopictus", "anopheles",
    "culex", "culiseta", "japonicus-koreicus"
}
clases_encontradas = set(df["class_label"].dropna().astype(str))

print("\nClases encontradas:", sorted(clases_encontradas))
print("Clases inesperadas:", sorted(clases_encontradas - clases_esperadas))

# Bounding boxes
cond_invalidas = (
    (df["img_w"] <= 0) | (df["img_h"] <= 0) |
    (df["bbx_xtl"] < 0) | (df["bbx_ytl"] < 0) |
    (df["bbx_xbr"] <= df["bbx_xtl"]) |
    (df["bbx_ybr"] <= df["bbx_ytl"]) |
    (df["bbx_xbr"] > df["img_w"]) |
    (df["bbx_ybr"] > df["img_h"])
)
invalidos = df[cond_invalidas]
print("Bounding boxes/dimensiones inválidas:", len(invalidos))
if len(invalidos):
    display(invalidos.head(10))

# Copia de trabajo: solo elimina duplicados exactos
df_clean = df.drop_duplicates().copy()

VALORES FALTANTES


,faltantes,porcentaje
img_fName,0,0.0
img_w,0,0.0
img_h,0,0.0
bbx_xtl,0,0.0
bbx_ytl,0,0.0
bbx_xbr,0,0.0
bbx_ybr,0,0.0
class_label,0,0.0


Total faltantes: 0
Filas duplicadas: 0
Nombres de imagen duplicados: 0

Imágenes en carpeta: 10357
Registros en CSV: 10357
Mencionadas en CSV pero inexistentes: 0
Imágenes sin anotación: 0

Clases encontradas: ['aegypti', 'albopictus', 'anopheles', 'culex', 'culiseta', 'japonicus-koreicus']
Clases inesperadas: []
Bounding boxes/dimensiones inválidas: 3


,img_fName,img_w,img_h,bbx_xtl,bbx_ytl,bbx_xbr,bbx_ybr,class_label
9004,train_09004.jpeg,1024,1365,435,233,1159,650,culex
10156,train_10156.jpeg,1024,2272,1070,512,1312,845,albopictus
10160,train_10160.jpeg,1816,4032,1755,522,2345,964,culex


# 5. Preprocesamiento e ingeniería de variables

Se crean variables derivadas para poder comparar imágenes de resoluciones diferentes:

- área y relación de aspecto de la imagen;
- ancho, alto, área y relación de aspecto del bounding box;
- proporción de la imagen ocupada por el mosquito;
- posición normalizada del centro del bounding box.

In [4]:
df_clean["image_area"] = df_clean["img_w"] * df_clean["img_h"]
df_clean["image_aspect_ratio"] = df_clean["img_w"] / df_clean["img_h"]

df_clean["bbox_width"] = df_clean["bbx_xbr"] - df_clean["bbx_xtl"]
df_clean["bbox_height"] = df_clean["bbx_ybr"] - df_clean["bbx_ytl"]
df_clean["bbox_area"] = df_clean["bbox_width"] * df_clean["bbox_height"]
df_clean["bbox_ratio"] = df_clean["bbox_area"] / df_clean["image_area"]
df_clean["bbox_aspect_ratio"] = df_clean["bbox_width"] / df_clean["bbox_height"]

df_clean["bbox_center_x_norm"] = (
    (df_clean["bbx_xtl"] + df_clean["bbx_xbr"]) / 2
) / df_clean["img_w"]

df_clean["bbox_center_y_norm"] = (
    (df_clean["bbx_ytl"] + df_clean["bbx_ybr"]) / 2
) / df_clean["img_h"]

print("Variables después del preprocesamiento:", df_clean.shape[1])
display(df_clean.head())

Variables después del preprocesamiento: 17


,img_fName,img_w,img_h,bbx_xtl,bbx_ytl,bbx_xbr,bbx_ybr,class_label,image_area,image_aspect_ratio,bbox_width,bbox_height,bbox_area,bbox_ratio,bbox_aspect_ratio,bbox_center_x_norm,bbox_center_y_norm
0,train_00000.jpeg,2448,3264,1301,1546,1641,2096,albopictus,7990272,0.750000,340,550,187000,0.023403,0.618182,0.600899,0.557904
1,train_00001.jpeg,3024,4032,900,1897,1950,2990,albopictus,12192768,0.750000,1050,1093,1147650,0.094125,0.960659,0.471230,0.606027
2,train_00002.jpeg,768,1024,220,58,659,808,albopictus,786432,0.750000,439,750,329250,0.418663,0.585333,0.572266,0.422852
3,train_00003.jpeg,3456,4608,1169,2364,1586,2826,albopictus,15925248,0.750000,417,462,192654,0.012097,0.902597,0.398582,0.563151
4,train_00004.jpeg,1024,1365,129,231,697,1007,culex,1397760,0.750183,568,776,440768,0.315339,0.731959,0.403320,0.453480
